# Fine-tune google/ddpm-cifar10-32 on the flowers (300 epochs)

Starts from the CIFAR-10 pretrained DDPM (35.7M) and fine-tunes it with the same data pipeline. Saves EMA checkpoints every 25 epochs.

In [1]:
!pip install -q torch-fidelity diffusers

In [2]:
RUN = "G_finetune"

BASE = dict(
    epochs=20, batch_size=32, lr=1e-4, warmup_steps=0,
    grad_clip=None,
    ema_decay=0.0,
    schedule="linear",
    arch="orig",
    dropout=0.0,
    amp=True,
    seed=0,
    save_every=0,
    eval_n=2000, eval_seeds=[0,1,2], eval_bs=250,
)

PRESETS = {
    "G_finetune": dict(grad_clip=1.0, ema_decay=0.9995, schedule="linear", epochs=300, warmup_steps=500, save_every=25, arch="pretrained", lr=5e-5),
}
CFG = {**BASE, **PRESETS[RUN]}
print(RUN, CFG)

G_finetune {'epochs': 300, 'batch_size': 32, 'lr': 5e-05, 'warmup_steps': 500, 'grad_clip': 1.0, 'ema_decay': 0.9995, 'schedule': 'linear', 'arch': 'pretrained', 'dropout': 0.0, 'amp': True, 'seed': 0, 'save_every': 25, 'eval_n': 2000, 'eval_seeds': [0, 1, 2], 'eval_bs': 250}


In [3]:
import os, json, math, time, copy, csv
import numpy as np, torch, torch.nn.functional as F, torchvision, kagglehub
import torchvision.transforms as T
from torchvision.utils import save_image
from tqdm.auto import tqdm
from diffusers import UNet2DModel

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = f"/kaggle/working/{RUN}"; os.makedirs(OUT, exist_ok=True)
json.dump(CFG, open(f"{OUT}/config.json", "w"), indent=2)
torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "", "| GPUs:", torch.cuda.device_count())

path = kagglehub.dataset_download("alxmamaev/flowers-recognition")
tf = T.Compose([T.ToTensor(), T.Resize((32, 32), interpolation=T.InterpolationMode.BICUBIC, antialias=True)])
ds = torchvision.datasets.ImageFolder(root=path, transform=tf)
X = torch.stack([ds[i][0] for i in range(len(ds))])
real_u8 = (X.clamp(0, 1) * 255).round().to(torch.uint8)
X_train = (X * 2 - 1).to(DEV)
N = len(X_train); print("images:", tuple(X.shape))

TSTEPS = 1000
def linear_betas(T_):
    s = 1000 / T_
    return torch.linspace(s * 1e-4, s * 0.02, T_, dtype=torch.float32)
def cosine_betas(T_, s=0.008):
    x = torch.arange(T_ + 1, dtype=torch.float64) / T_
    ab = torch.cos((x + s) / (1 + s) * math.pi / 2) ** 2
    ab = ab / ab[0]
    return (1 - ab[1:] / ab[:-1]).clamp(max=0.999).float()

beta = (cosine_betas if CFG["schedule"] == "cosine" else linear_betas)(TSTEPS).to(DEV)
alpha = 1 - beta
a_bar = torch.cumprod(alpha, 0)
sqrt_ab, sqrt_1m_ab = a_bar.sqrt(), (1 - a_bar).sqrt()
one_by_sqrt_alpha = 1 / alpha.sqrt()

def build_model(arch, dropout):
    if arch == "pretrained":
        return UNet2DModel.from_pretrained("google/ddpm-cifar10-32")
    if arch == "orig":
        down = ("DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D")
        up   = ("AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D")
    elif arch == "attn16":
        down = ("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D", "DownBlock2D")
        up   = ("UpBlock2D", "AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D")
    return UNet2DModel(in_channels=3, out_channels=3, block_out_channels=[64, 128, 256, 256],
                       layers_per_block=2, down_block_types=down, up_block_types=up, dropout=dropout)

model = build_model(CFG["arch"], CFG["dropout"]).to(DEV)
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params/1e6:.2f}M")

device: cuda Tesla T4 | GPUs: 2
images: (4317, 3, 32, 32)


params: 35.75M


In [4]:
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"])
scaler = torch.amp.GradScaler(DEV.type, enabled=CFG["amp"] and DEV.type == "cuda")
ema = copy.deepcopy(model).eval().requires_grad_(False) if CFG["ema_decay"] > 0 else None

@torch.no_grad()
def ema_update(step):
    d = min(CFG["ema_decay"], (1 + step) / (10 + step))
    for pe, p in zip(ema.parameters(), model.parameters()):
        pe.lerp_(p, 1 - d)
    for be, b in zip(ema.buffers(), model.buffers()):
        be.copy_(b)

log, step, t_start = [], 0, time.time()
for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    perm = torch.randperm(N, device=DEV)
    losses, nan_steps, max_gn = [], 0, 0.0
    for i in range(0, N, CFG["batch_size"]):
        x0 = X_train[perm[i:i + CFG["batch_size"]]]
        flip = torch.rand(len(x0), 1, 1, 1, device=DEV) < 0.5
        x0 = torch.where(flip, x0.flip(3), x0)
        t = torch.randint(1, TSTEPS, (len(x0),), device=DEV)
        noise = torch.randn_like(x0)
        xt = sqrt_ab[t].view(-1, 1, 1, 1) * x0 + sqrt_1m_ab[t].view(-1, 1, 1, 1) * noise

        if CFG["warmup_steps"]:
            for g in opt.param_groups:
                g["lr"] = CFG["lr"] * min(1.0, (step + 1) / CFG["warmup_steps"])
        with torch.autocast(DEV.type, dtype=torch.float16, enabled=CFG["amp"] and DEV.type == "cuda"):
            pred = model(xt, t).sample
            loss = F.mse_loss(pred.float(), noise)

        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if CFG["grad_clip"]:
            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            if torch.isfinite(gn): max_gn = max(max_gn, gn.item())
        scaler.step(opt); scaler.update()
        if ema is not None: ema_update(step)

        lv = loss.item()
        if math.isfinite(lv): losses.append(lv)
        else: nan_steps += 1
        step += 1

    row = dict(epoch=epoch, step=step, loss=float(np.mean(losses)) if losses else float("nan"),
               nan_steps=nan_steps, max_grad_norm=round(max_gn, 3), minutes=round((time.time() - t_start) / 60, 2))
    log.append(row); print(row)
    if CFG["save_every"] and epoch % CFG["save_every"] == 0:
        torch.save((ema or model).state_dict(), f"{OUT}/weights_ep{epoch}.pt")

train_minutes = (time.time() - t_start) / 60
torch.save(model.state_dict(), f"{OUT}/model.pt")
if ema is not None: torch.save(ema.state_dict(), f"{OUT}/ema.pt")
with open(f"{OUT}/train_log.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(log[0])); w.writeheader(); w.writerows(log)
print(f"training done: {train_minutes:.1f} min, total NaN steps: {sum(r['nan_steps'] for r in log)}")

{'epoch': 1, 'step': 135, 'loss': 0.042021523885152956, 'nan_steps': 0, 'max_grad_norm': 0.038, 'minutes': 0.41}
{'epoch': 2, 'step': 270, 'loss': 0.04142225398509591, 'nan_steps': 0, 'max_grad_norm': 0.035, 'minutes': 0.79}
{'epoch': 3, 'step': 405, 'loss': 0.04513938403515904, 'nan_steps': 0, 'max_grad_norm': 0.038, 'minutes': 1.18}
{'epoch': 4, 'step': 540, 'loss': 0.04147652683572637, 'nan_steps': 0, 'max_grad_norm': 0.037, 'minutes': 1.58}
{'epoch': 5, 'step': 675, 'loss': 0.04413690252436532, 'nan_steps': 0, 'max_grad_norm': 0.051, 'minutes': 1.97}
{'epoch': 6, 'step': 810, 'loss': 0.04154717455859538, 'nan_steps': 0, 'max_grad_norm': 0.032, 'minutes': 2.35}
{'epoch': 7, 'step': 945, 'loss': 0.042746236382259266, 'nan_steps': 0, 'max_grad_norm': 0.034, 'minutes': 2.75}
{'epoch': 8, 'step': 1080, 'loss': 0.04101384106195635, 'nan_steps': 0, 'max_grad_norm': 0.038, 'minutes': 3.14}
{'epoch': 9, 'step': 1215, 'loss': 0.04131006014153913, 'nan_steps': 0, 'max_grad_norm': 0.035, 'minu

In [5]:
import sys
for m in [k for k in list(sys.modules) if k.startswith("torchmetrics")]:
    del sys.modules[m]
import torch_fidelity
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

net = (ema or model).eval()
net_dp = torch.nn.DataParallel(net) if torch.cuda.device_count() > 1 else net

@torch.no_grad()
def sample(n, seed, bs):
    g = torch.Generator(device=DEV).manual_seed(seed); outs = []
    for s in range(0, n, bs):
        b = min(bs, n - s)
        x = torch.randn((b, 3, 32, 32), device=DEV, generator=g)
        for t in tqdm(reversed(range(1, TSTEPS)), total=TSTEPS - 1, desc=f"batch {s//bs+1}/{(n+bs-1)//bs}", leave=False):
            ts = torch.full((b,), t, dtype=torch.long, device=DEV)
            z = torch.randn(x.shape, device=DEV, generator=g) if t > 1 else torch.zeros_like(x)
            eps = net_dp(x, ts, return_dict=False)[0]
            x = one_by_sqrt_alpha[t] * (x - (beta[t] / sqrt_1m_ab[t]) * eps) + beta[t].sqrt() * z
        x = torch.nan_to_num(x, nan=0.0).clamp(-1, 1)
        outs.append(((x + 1) / 2 * 255).round().to(torch.uint8).cpu())
    return torch.cat(outs)

def fid_kid(a_u8, b_u8):
    fid = FrechetInceptionDistance(feature=2048).to(DEV)
    kid = KernelInceptionDistance(subset_size=min(1000, len(a_u8), len(b_u8))).to(DEV)
    for x, flag in ((a_u8, True), (b_u8, False)):
        for i in range(0, len(x), 500):
            xb = x[i:i + 500].to(DEV); fid.update(xb, real=flag); kid.update(xb, real=flag)
    km, ks = kid.compute()
    return fid.compute().item(), km.item(), ks.item()

def nn_check(gen_u8, k=64):
    R = real_u8.float().flatten(1) / 255; G = gen_u8[:k].float().flatten(1) / 255
    exact = "donot_use_mm_for_euclid_dist"
    d_gen, idx = torch.cdist(G, R, compute_mode=exact).min(1)
    q = torch.randperm(len(R), generator=torch.Generator().manual_seed(0))[:k]
    dr = torch.cdist(R[q], R, compute_mode=exact)
    dr[torch.arange(k), q] = float("inf")
    pairs = torch.stack([gen_u8[:k], real_u8[idx]], 1).flatten(0, 1).float() / 255
    save_image(pairs[:32], f"{OUT}/nearest_neighbours.png", nrow=8)
    return d_gen.mean().item(), dr.min(1).values.mean().item()

res = dict(run=RUN, cfg=CFG, params_M=round(n_params / 1e6, 2), train_minutes=round(train_minutes, 1),
           total_nan_steps=sum(r["nan_steps"] for r in log), gpu=torch.cuda.get_device_name(0) if DEV.type == "cuda" else "cpu",
           evaluated="ema" if ema is not None else "raw", runs=[])
for sd in CFG["eval_seeds"]:
    t0 = time.time()
    gen = sample(CFG["eval_n"], sd, CFG["eval_bs"])
    np.save(f"{OUT}/samples_seed{sd}.npy", gen.numpy())
    if sd == CFG["eval_seeds"][0]:
        save_image(gen[:64].float() / 255, f"{OUT}/samples_grid.png", nrow=8)
        res["nn_dist_gen"], res["nn_dist_real"] = nn_check(gen)
    f, km, ks = fid_kid(real_u8, gen)
    res["runs"].append(dict(seed=sd, fid=f, kid=km, kid_std=ks, sample_minutes=round((time.time() - t0) / 60, 1)))
    print(res["runs"][-1])
    json.dump(res, open(f"{OUT}/results.json", "w"), indent=2)

fids = [r["fid"] for r in res["runs"]]; kids = [r["kid"] for r in res["runs"]]
res.update(fid_mean=float(np.mean(fids)), fid_std=float(np.std(fids)),
           kid_mean=float(np.mean(kids)), kid_std_seeds=float(np.std(kids)))
json.dump(res, open(f"{OUT}/results.json", "w"), indent=2)
print(json.dumps({k: v for k, v in res.items() if k not in ("runs", "cfg")}, indent=2))

> **Note:** this evaluation cell failed on Kaggle (torch.nn.DataParallel does not work with this diffusers UNet). The trained weights were already saved; evaluation was redone in `02_eval_reference_sampler.ipynb`.